# Fine-tuning PaddleOCR CRNN cho nhận dạng chữ tiếng Nhật (Manga109-s)

**Pipeline:**

1. Cài đặt môi trường – PaddlePaddle-GPU + PaddleOCR
2. Chuẩn bị dữ liệu – manga109 rec data (SimpleDataSet format)
3. Tạo dictionary – trích ký tự từ label files
4. Tải pretrained weights – Chinese PP-OCR v2.0 Mobile Rec (CRNN, cùng kiến trúc)
5. Tạo config YAML – auto-resume nếu có checkpoint
6. Training – distributed 2×GPU (T4)
7. Export inference model
8. Test inference

**Kiến trúc:** CRNN = MobileNetV3-Small + BiLSTM + CTC

**Kaggle:** 2×T4 GPU, ~30 GB disk

---
## 1. Cài đặt môi trường

In [ ]:
import subprocess, sys, os

# Kaggle thường cài sẵn torch + nvidia-cu12 (CUDA 12.8).
# Khi cài Paddle sẽ kéo bộ nvidia-* khác version -> conflict warning.
# Notebook này chỉ dùng PaddleOCR nên gỡ stack torch trước để tránh xung đột.
subprocess.run([
    sys.executable, '-m', 'pip', 'uninstall', '-y',
    'torch', 'torchvision', 'torchaudio', 'triton',
    'fastai', 'fastcore', 'fastdownload'
], check=False)

def get_cuda_major_version():
    try:
        out = subprocess.check_output(['nvcc', '--version'], text=True)
        for tok in out.split():
            if tok.startswith('V'):
                return tok.split('.')[0].replace('V', '')
    except Exception:
        pass
    return '11'

cuda_ver = get_cuda_major_version()
cu_tag = 'cu118' if cuda_ver == '11' else 'cu126'
print(f'Detected CUDA {cuda_ver} -> installing paddlepaddle-gpu for {cu_tag}')

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'pip', 'setuptools', 'wheel'
])

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'paddlepaddle-gpu==3.0.0',
    '-i', f'https://www.paddlepaddle.org.cn/packages/stable/{cu_tag}/'
])

print('Paddle installation completed.')

In [ ]:
import paddle
print('PaddlePaddle version:', paddle.__version__)
print('GPU available:', paddle.device.is_compiled_with_cuda())
print('GPU count:', paddle.device.cuda.device_count())
for i in range(paddle.device.cuda.device_count()):
    print(f'  GPU {i}: {paddle.device.cuda.get_device_name(i)}')

In [ ]:
PADDLEOCR_DIR = '/kaggle/working/PaddleOCR'

if not os.path.isdir(PADDLEOCR_DIR):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git {PADDLEOCR_DIR}

!pip install -q -r {PADDLEOCR_DIR}/requirements.txt
print('PaddleOCR ready at', PADDLEOCR_DIR)

---
## 2. Chuẩn bị dữ liệu

Pipeline giống `jap-ocr-vcrnn.ipynb`:

1. Lấy HF token từ **Kaggle Secrets**
2. Download `hal-utokyo/Manga109-s` từ Hugging Face
3. Giải nén zip (~3 GB) → `Manga109s_released_2023_12_07/`
4. Parse toàn bộ **annotation XML**, crop từng text box thành ảnh PNG
5. Lưu theo format **PaddleOCR SimpleDataSet**:

```
/kaggle/working/train_data/manga109_rec/
├── train/          ← ảnh crop (word_000001.png ...)
├── val/
├── rec_gt_train.txt    ← train/word_000001.png\ttext
└── rec_gt_val.txt
```

In [ ]:
from kaggle_secrets import UserSecretsClient

secret_label = "HF_TOKEN"
secret_value = UserSecretsClient().get_secret(secret_label)

WORK_DIR        = '/kaggle/working'
HF_DOWNLOAD_DIR = f'{WORK_DIR}/manga109s_dataset'
EXTRACT_DIR     = f'{WORK_DIR}/manga109_extracted'
DATASET_ROOT    = f'{EXTRACT_DIR}/Manga109s_released_2023_12_07'
DATA_DIR        = f'{WORK_DIR}/train_data/manga109_rec'   # output của bước crop
OUTPUT_DIR      = f'{WORK_DIR}/output/rec_japan_crnn'
DICT_PATH       = f'{WORK_DIR}/japan_manga_dict.txt'
CONFIG_PATH     = f'{WORK_DIR}/rec_japan_crnn_ft.yml'
PRETRAINED_DIR  = f'{WORK_DIR}/pretrained'

os.makedirs(HF_DOWNLOAD_DIR, exist_ok=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PRETRAINED_DIR, exist_ok=True)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])

In [ ]:
from huggingface_hub import login, snapshot_download
import zipfile

print('Đăng nhập Hugging Face bằng Kaggle Secret...')
login(secret_value)

zip_path = f'{HF_DOWNLOAD_DIR}/Manga109s_released_2023_12_07.zip'

if not os.path.isfile(zip_path):
    print('Đang tải Manga109-s từ Hugging Face...')
    subprocess.run(['rm', '-rf',
        '/root/.cache/huggingface/datasets',
        f'{WORK_DIR}/hf_cache',
    ], check=False)
    snapshot_download(
        repo_id='hal-utokyo/Manga109-s',
        repo_type='dataset',
        local_dir=HF_DOWNLOAD_DIR,
        token=secret_value,
    )
    print(f'Tải xong. Files: {os.listdir(HF_DOWNLOAD_DIR)}')
else:
    print('Đã có file zip, bỏ qua bước tải.')

if not os.path.isdir(DATASET_ROOT) or not os.listdir(DATASET_ROOT):
    print('Đang giải nén (~3 GB)...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print('Giải nén xong!')
else:
    print('Dataset đã giải nén từ trước, bỏ qua.')

print(f'Nội dung dataset: {os.listdir(DATASET_ROOT)}')

In [ ]:
import xml.etree.ElementTree as ET
import random
from PIL import Image
from typing import Optional
from tqdm import tqdm


class Manga109RecDatasetCreator:
    """
    Tạo dataset Text Recognition (PaddleOCR SimpleDataSet format) từ Manga109-s.
    Duyệt annotation XML, crop từng text box thành ảnh riêng, kèm label file train/val.
    """

    def __init__(self, dataset_root: str):
        self.dataset_root    = dataset_root
        self.annotations_dir = os.path.join(dataset_root, 'annotations')
        self.images_dir      = os.path.join(dataset_root, 'images')

    def create_rec_dataset(
        self,
        output_dir:  str,
        min_width:   int = 20,
        min_height:  int = 20,
        max_samples: Optional[int] = 50_000,
        train_ratio: float = 0.9,
        seed:        int = 42,
    ):
        random.seed(seed)

        train_img_dir = os.path.join(output_dir, 'train')
        val_img_dir   = os.path.join(output_dir, 'val')
        os.makedirs(train_img_dir, exist_ok=True)
        os.makedirs(val_img_dir,   exist_ok=True)

        train_gt = os.path.join(output_dir, 'rec_gt_train.txt')
        val_gt   = os.path.join(output_dir, 'rec_gt_val.txt')

        xml_files = sorted(f for f in os.listdir(self.annotations_dir) if f.endswith('.xml'))
        print(f'Tìm thấy {len(xml_files)} truyện. Đang thu thập samples...')

        all_samples = []

        for xml_file in tqdm(xml_files, desc='Parsing books'):
            book_title = os.path.splitext(xml_file)[0]
            xml_path   = os.path.join(self.annotations_dir, xml_file)
            try:
                root = ET.parse(xml_path).getroot()
            except Exception:
                continue

            for page in root.findall('.//page'):
                page_idx     = page.get('index')
                img_filename = f'{int(page_idx):03d}.jpg'
                img_path     = os.path.join(self.images_dir, book_title, img_filename)
                if not os.path.exists(img_path):
                    continue
                try:
                    img = Image.open(img_path).convert('RGB')
                except Exception:
                    continue

                for text_obj in page.findall('text'):
                    try:
                        xmin  = int(text_obj.get('xmin'))
                        ymin  = int(text_obj.get('ymin'))
                        xmax  = int(text_obj.get('xmax'))
                        ymax  = int(text_obj.get('ymax'))
                        label = (text_obj.text or '').strip()
                        label = label.replace('\n', '').replace('\t', ' ')
                        if not label:
                            continue
                        if (xmax - xmin) < min_width or (ymax - ymin) < min_height:
                            continue
                        all_samples.append((img.crop((xmin, ymin, xmax, ymax)), label))
                    except Exception:
                        continue

            # Dừng sớm khi đủ mẫu (thu dư 2× để shuffle đều)
            if max_samples and len(all_samples) >= max_samples * 2:
                break

        random.shuffle(all_samples)
        if max_samples:
            all_samples = all_samples[:max_samples]

        split_idx     = int(len(all_samples) * train_ratio)
        train_samples = all_samples[:split_idx]
        val_samples   = all_samples[split_idx:]
        print(f'\nSplit: {len(train_samples):,} train | {len(val_samples):,} val')

        def _write_split(samples, img_dir, gt_path, split_name, start_idx=1):
            with open(gt_path, 'w', encoding='utf-8') as gt_file:
                for i, (crop, label) in enumerate(
                    tqdm(samples, desc=f'Saving {split_name}'), start=start_idx
                ):
                    img_name  = f'word_{i:06d}.png'
                    save_path = os.path.join(img_dir, img_name)
                    try:
                        crop.save(save_path)
                    except Exception:
                        continue
                    gt_file.write(f'{split_name}/{img_name}\t{label}\n')
            return len(samples)

        n_train = _write_split(train_samples, train_img_dir, train_gt, 'train', start_idx=1)
        n_val   = _write_split(val_samples,   val_img_dir,   val_gt,   'val',   start_idx=n_train + 1)

        print(f'\nHOÀN THÀNH Recognition Dataset!')
        print(f'   Train : {n_train:,} ảnh  →  {train_gt}')
        print(f'   Val   : {n_val:,} ảnh  →  {val_gt}')
        return n_train, n_val


# Chỉ chạy crop nếu chưa có label file (idempotent)
train_label = os.path.join(DATA_DIR, 'rec_gt_train.txt')
val_label   = os.path.join(DATA_DIR, 'rec_gt_val.txt')

if os.path.isfile(train_label) and os.path.isfile(val_label):
    print('Rec dataset đã tồn tại, bỏ qua bước crop.')
else:
    creator = Manga109RecDatasetCreator(DATASET_ROOT)
    creator.create_rec_dataset(
        output_dir  = DATA_DIR,
        max_samples = 50_000,
        train_ratio = 0.9,
        seed        = 42,
    )

def count_lines(path):
    with open(path, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)

n_train = count_lines(train_label)
n_val   = count_lines(val_label)
print(f'\nTrain samples: {n_train:,}')
print(f'Val   samples: {n_val:,}')

with open(train_label, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(repr(line.strip()))

---
## 3. Tạo dictionary từ labels

Trích tất cả ký tự duy nhất từ train + val labels, sắp xếp theo Unicode.

In [ ]:
def build_dict_from_labels(*label_files):
    chars = set()
    max_len = 0
    for path in label_files:
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t', 1)
                if len(parts) < 2:
                    continue
                text = parts[1]
                chars.update(text)
                max_len = max(max_len, len(text))
    return sorted(chars), max_len

all_chars, max_text_len = build_dict_from_labels(train_label, val_label)
print(f'Unique characters: {len(all_chars)}')
print(f'Max text length  : {max_text_len}')
print(f'Sample chars     : {all_chars[:30]}')

with open(DICT_PATH, 'w', encoding='utf-8') as f:
    for ch in all_chars:
        f.write(ch + '\n')

print(f'\nDictionary saved to {DICT_PATH} ({len(all_chars)} chars)')

MAX_TEXT_LENGTH = min(max_text_len + 5, 80)
print(f'Config max_text_length = {MAX_TEXT_LENGTH}')

---
## 4. Tải pretrained weights

Sử dụng **ch_ppocr_mobile_v2.0_rec_train** (CRNN – cùng kiến trúc MobileNetV3-Small + BiLSTM + CTC).  
Khi finetune, backbone + neck weights được load, Head (FC cuối) sẽ tự khởi tạo lại do số class khác.

In [ ]:
PRETRAINED_URL  = 'https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_rec_train.tar'
PRETRAINED_TAR  = os.path.join(PRETRAINED_DIR, 'ch_ppocr_mobile_v2.0_rec_train.tar')
PRETRAINED_PATH = os.path.join(PRETRAINED_DIR, 'ch_ppocr_mobile_v2.0_rec_train', 'best_accuracy')

if not os.path.isfile(PRETRAINED_TAR):
    print('Downloading pretrained weights...')
    !wget -q --show-progress -O {PRETRAINED_TAR} {PRETRAINED_URL}

if not os.path.isdir(os.path.dirname(PRETRAINED_PATH)):
    !tar -xf {PRETRAINED_TAR} -C {PRETRAINED_DIR}

print('Pretrained model path:', PRETRAINED_PATH)
!ls -lh {os.path.dirname(PRETRAINED_PATH)}

---
## 5. Tạo config YAML

Kiến trúc CRNN:
- **Backbone:** MobileNetV3 (small, scale=0.5)
- **Neck:** BiLSTM (hidden=48)
- **Head:** CTC

Hỗ trợ **auto-resume**: nếu có checkpoint trong `OUTPUT_DIR`, tự động load lại.

In [ ]:
import glob

def find_latest_checkpoint(output_dir):
    """Tìm checkpoint mới nhất (latest.pdparams) trong output_dir."""
    latest = os.path.join(output_dir, 'latest')
    if os.path.isfile(latest + '.pdparams'):
        return latest
    return None

checkpoint = find_latest_checkpoint(OUTPUT_DIR)
pretrained = '' if checkpoint else PRETRAINED_PATH

if checkpoint:
    print(f'AUTO-RESUME: Tìm thấy checkpoint → {checkpoint}')
else:
    print(f'Không có checkpoint, dùng pretrained: {pretrained}')

In [ ]:
import yaml

N_GPUS = paddle.device.cuda.device_count()
BATCH_PER_CARD = 256
STEPS_PER_EPOCH = max(1, n_train // (BATCH_PER_CARD * N_GPUS))

config = {
    'Global': {
        'use_gpu': True,
        'epoch_num': 200,
        'log_smooth_window': 20,
        'print_batch_step': 10,
        'save_model_dir': OUTPUT_DIR,
        'save_epoch_step': 10,
        'eval_batch_step': [0, STEPS_PER_EPOCH * 5],
        'cal_metric_during_train': True,
        'pretrained_model': pretrained,
        'checkpoints': checkpoint or '',
        'save_inference_dir': f'{OUTPUT_DIR}/inference',
        'use_visualdl': False,
        'infer_img': '',
        'character_dict_path': DICT_PATH,
        'max_text_length': MAX_TEXT_LENGTH,
        'infer_mode': False,
        'use_space_char': True,
        'distributed': True,
        'save_res_path': f'{OUTPUT_DIR}/predicts.txt',
        'seed': 2024,
    },
    'Optimizer': {
        'name': 'Adam',
        'beta1': 0.9,
        'beta2': 0.999,
        'lr': {
            'name': 'Cosine',
            'learning_rate': 0.001,
            'warmup_epoch': 5,
        },
        'regularizer': {
            'name': 'L2',
            'factor': 1.0e-5,
        },
    },
    'Architecture': {
        'model_type': 'rec',
        'algorithm': 'CRNN',
        'Transform': None,
        'Backbone': {
            'name': 'MobileNetV3',
            'scale': 0.5,
            'model_name': 'small',
            'small_stride': [1, 2, 2, 2],
        },
        'Neck': {
            'name': 'SequenceEncoder',
            'encoder_type': 'rnn',
            'hidden_size': 48,
        },
        'Head': {
            'name': 'CTCHead',
            'fc_decay': 1.0e-5,
        },
    },
    'Loss': {'name': 'CTCLoss'},
    'PostProcess': {'name': 'CTCLabelDecode'},
    'Metric': {
        'name': 'RecMetric',
        'main_indicator': 'acc',
    },
    'Train': {
        'dataset': {
            'name': 'SimpleDataSet',
            'data_dir': DATA_DIR,
            'label_file_list': [os.path.join(DATA_DIR, 'rec_gt_train.txt')],
            'transforms': [
                {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
                {'RecAug': None},
                {'CTCLabelEncode': None},
                {'RecResizeImg': {'image_shape': [3, 32, 320]}},
                {'KeepKeys': {'keep_keys': ['image', 'label', 'length']}},
            ],
        },
        'loader': {
            'shuffle': True,
            'batch_size_per_card': BATCH_PER_CARD,
            'drop_last': True,
            'num_workers': 4,
            'use_shared_memory': True,
        },
    },
    'Eval': {
        'dataset': {
            'name': 'SimpleDataSet',
            'data_dir': DATA_DIR,
            'label_file_list': [os.path.join(DATA_DIR, 'rec_gt_val.txt')],
            'transforms': [
                {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
                {'CTCLabelEncode': None},
                {'RecResizeImg': {'image_shape': [3, 32, 320]}},
                {'KeepKeys': {'keep_keys': ['image', 'label', 'length']}},
            ],
        },
        'loader': {
            'shuffle': False,
            'drop_last': False,
            'batch_size_per_card': BATCH_PER_CARD,
            'num_workers': 4,
            'use_shared_memory': True,
        },
    },
}

with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f'Config saved to {CONFIG_PATH}')
print(f'Steps per epoch ≈ {STEPS_PER_EPOCH}')
print(f'Eval every {STEPS_PER_EPOCH * 5} steps ≈ 5 epochs')
print()
with open(CONFIG_PATH, 'r') as f:
    print(f.read())

---
## 6. Training (Distributed 2×GPU)

Sử dụng `paddle.distributed.launch` để chạy trên 2 GPU T4.

In [ ]:
gpu_list = ','.join(str(i) for i in range(N_GPUS))

train_cmd = (
    f'python -m paddle.distributed.launch '
    f'--gpus="{gpu_list}" '
    f'{PADDLEOCR_DIR}/tools/train.py '
    f'-c {CONFIG_PATH}'
)

print('Training command:')
print(train_cmd)
print()
print(f'Logs will be saved to {OUTPUT_DIR}')

In [ ]:
!{train_cmd}

---
## 7. Kết quả training

Kiểm tra model đã lưu và log.

In [ ]:
print('=== Saved models ===')
!ls -lh {OUTPUT_DIR}/*.pdparams 2>/dev/null | tail -10

best_model = os.path.join(OUTPUT_DIR, 'best_accuracy')
if os.path.isfile(best_model + '.pdparams'):
    print(f'\nBest model: {best_model}')
    !ls -lh {best_model}.*

train_result = os.path.join(OUTPUT_DIR, 'train_result.json')
if os.path.isfile(train_result):
    import json
    with open(train_result, 'r') as f:
        result = json.load(f)
    print(f'\n=== Training Result ===')
    for k, v in result.items():
        print(f'  {k}: {v}')

---
## 8. Export Inference Model

Chuyển dynamic graph model → static graph model để deploy.

In [ ]:
INFERENCE_DIR = f'{OUTPUT_DIR}/inference'

export_cmd = (
    f'python {PADDLEOCR_DIR}/tools/export_model.py '
    f'-c {CONFIG_PATH} '
    f'-o Global.pretrained_model={best_model} '
    f'Global.save_inference_dir={INFERENCE_DIR}'
)

print('Export command:')
print(export_cmd)
!{export_cmd}

print('\n=== Inference model files ===')
!ls -lh {INFERENCE_DIR}/

---
## 9. Test Inference

Chạy thử nhận dạng trên vài ảnh val.

In [ ]:
import random

with open(val_label, 'r', encoding='utf-8') as f:
    val_lines = [l.strip() for l in f if l.strip()]

sample_lines = random.sample(val_lines, min(10, len(val_lines)))

sample_imgs = []
sample_gts  = []
for line in sample_lines:
    parts = line.split('\t', 1)
    img_path = os.path.join(DATA_DIR, parts[0])
    gt_text  = parts[1] if len(parts) > 1 else ''
    sample_imgs.append(img_path)
    sample_gts.append(gt_text)

print(f'Selected {len(sample_imgs)} sample images for inference test')

In [ ]:
import sys
sys.path.insert(0, PADDLEOCR_DIR)

from tools.infer_rec import main as infer_main
from ppocr.utils.utility import get_image_file_list

for img_path, gt in zip(sample_imgs, sample_gts):
    infer_cmd = (
        f'python {PADDLEOCR_DIR}/tools/infer_rec.py '
        f'-c {CONFIG_PATH} '
        f'-o Global.pretrained_model={best_model} '
        f'Global.infer_img={img_path}'
    )
    result = subprocess.run(infer_cmd, shell=True, capture_output=True, text=True)
    pred_lines = [l for l in result.stdout.strip().split('\n') if 'result:' in l.lower() or 'infer_img' in l.lower()]
    pred_text = pred_lines[-1] if pred_lines else result.stdout.strip().split('\n')[-1]

    print(f'Image: {os.path.basename(img_path)}')
    print(f'  GT  : {gt}')
    print(f'  Pred: {pred_text}')
    print()

---
## 10. Lưu artifacts (tải về từ Kaggle)

Copy model + dict vào `/kaggle/working/` để tải về.

In [ ]:
import shutil

ARTIFACT_DIR = '/kaggle/working/artifacts'
os.makedirs(ARTIFACT_DIR, exist_ok=True)

shutil.copy2(DICT_PATH, ARTIFACT_DIR)
shutil.copy2(CONFIG_PATH, ARTIFACT_DIR)

for ext in ['.pdparams', '.pdopt', '.states']:
    src = best_model + ext
    if os.path.isfile(src):
        shutil.copy2(src, os.path.join(ARTIFACT_DIR, 'best_accuracy' + ext))

inf_dst = os.path.join(ARTIFACT_DIR, 'inference')
if os.path.isdir(INFERENCE_DIR):
    if os.path.isdir(inf_dst):
        shutil.rmtree(inf_dst)
    shutil.copytree(INFERENCE_DIR, inf_dst)

if os.path.isfile(train_result):
    shutil.copy2(train_result, ARTIFACT_DIR)

print('=== Artifacts ===')
!find {ARTIFACT_DIR} -type f | head -20
!du -sh {ARTIFACT_DIR}